# Lab: inferring what we cannot see — an ice-shelf inverse problem

```{note}
This lab accompanies {doc}`icepack-ice-shelf` and draws on the rheology discussion in {doc}`../ice_flow/ice-rheology`. You will need the Firedrake/icepack environment from the course container, running the **Firedrake (icepack)** kernel.
```

We can measure ice velocity from space. We cannot drill through an ice shelf and directly measure the material property that governs how it flows — the fluidity $A$ in Glen's flow law, $\dot\varepsilon = A\,\tau^n$. That asymmetry, observable velocity but hidden material property, is the setup for an **inverse problem**.

This lab works entirely with synthetic data, so we will know the truth and can grade the recovery honestly. The workflow has four acts: build a synthetic shelf with a known, spatially varying fluidity; solve the forward problem for velocity and corrupt the result with noise; pose and solve the inverse problem to estimate the fluidity from the noisy velocity; and finally study what happens when the regularization is wrong. The same pattern — forward model, synthetic twin, inversion, sensitivity study — is how groups like those behind [TODO-CITE: Morlighem ice-shelf inversion] and [TODO-CITE: Goldberg-Heimbach adjoint method] validate their methods before touching real data.

Everything here is a **diagnostic** experiment: we solve only the momentum balance, not the time-dependent mass conservation. That keeps the problem tractable for a lab and isolates the core inverse logic from numerical time-stepping details.

## Setup

We import Firedrake and icepack, then define a few constants we will reuse throughout. The domain is a rectangle of width $L_x = 50$ km and $L_y = 20$ km — wide enough that a soft band running across the shelf will have room to contrast with the stiffer surroundings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import firedrake
import icepack
import icepack.statistics  # verify against your icepack version

# Domain dimensions
Lx = 50e3     # along-flow length, m
Ly = 20e3     # cross-flow width, m
nx, ny = 50, 20

mesh = firedrake.RectangleMesh(nx, ny, Lx, Ly)
Q = firedrake.FunctionSpace(mesh, "CG", 2)   # scalar fields
V = firedrake.VectorFunctionSpace(mesh, "CG", 2)   # vector fields

x, y = firedrake.SpatialCoordinate(mesh)
print("Mesh ready:", mesh.num_cells(), "cells")

## Part 1 — the synthetic truth

### Thickness and boundary conditions

We set up the same geometry as {doc}`icepack-ice-shelf`: a thickness that thins linearly downstream from 500 m at the inflow ($x=0$) to 400 m at the calving front ($x=L_x$), and an inflow velocity of 100 m yr$^{-1}$ uniform across the upstream face. Boundary ids follow the icepack convention: `1` is the inflow, `2` the calving front (free), `3` and `4` the lateral margins (also free for a shelf).

### The prescribed fluidity — a soft Gaussian band

Here is where the lab departs from the chapter. Instead of a spatially uniform temperature, we prescribe a **spatially varying fluidity** that has a warm, soft streak running across the shelf parallel to the $y$-axis. Physically, you can think of this as a band of softer ice — perhaps a relict shear margin or a region of marine meltwater infiltration — centred at $x_0 = 0.4\,L_x$ with along-flow half-width $\sigma_x = 4$ km and cross-flow half-width $\sigma_y = 3$ km, raised to a peak fluidity $A_\text{peak}$ that is 10 times the background $A_0$.

We express fluidity in log space throughout. Defining $\theta = \log(A / A_0)$, the true log-fluidity is

$$\theta_\text{true}(x,y) = \log\!\left(1 + 9\,\exp\!\left(-\frac{(x-x_0)^2}{2\sigma_x^2} - \frac{(y-y_c)^2}{2\sigma_y^2}\right)\right)$$

where $y_c = 0.45\,L_y$ offsets the band slightly from the centreline so it is not perfectly symmetric. The factor of 9 makes the peak $A/A_0 = 10$ while the tails approach $A/A_0 = 1$, i.e. $\theta \to 0$. Working in log space keeps the fluidity positive everywhere and is the natural parameterisation for the inverse problem, because the prior uncertainty is more nearly Gaussian in $\log A$ than in $A$ itself.

In [ ]:
# Reference fluidity at a uniform cold-shelf temperature (-18 °C = 255 K)
T_ref = firedrake.Constant(255.0)
A0 = icepack.rate_factor(T_ref)   # Pa^-3 yr^-1, scalar Constant

# Thickness: linear ramp 500 m → 400 m downstream
h_in = 500.0
h = firedrake.Function(Q).interpolate(h_in - 100.0 * x / Lx)

# Prescribed log-fluidity: background = 0, Gaussian soft band
x0 = 0.4 * Lx
yc = 0.45 * Ly
sigma_x = 4e3
sigma_y = 3e3

theta_true_expr = firedrake.ln(
    1.0 + 9.0 * firedrake.exp(
        -((x - x0)**2 / (2 * sigma_x**2))
        - ((y - yc)**2 / (2 * sigma_y**2))
    )
)
theta_true = firedrake.Function(Q).interpolate(theta_true_expr)

# True fluidity field: A = A0 * exp(theta)
A_true = firedrake.Function(Q).interpolate(A0 * firedrake.exp(theta_true))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (field, label) in zip(axes, [(theta_true, 'θ_true'), (A_true, 'A_true (Pa⁻³ yr⁻¹)')]):
    ax.set_aspect('equal')
    colors = firedrake.tripcolor(field, axes=ax)
    fig.colorbar(colors, ax=ax, label=label)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
plt.tight_layout()
plt.show()

## Part 2 — the forward solve and synthetic observations

### Solving for velocity with the true fluidity

We use the same `IceShelf` model as in {doc}`icepack-ice-shelf`. The Dirichlet condition is applied only at the inflow edge (`dirichlet_ids=[1]`); the calving front and lateral margins are traction-free. The diagnostic solve minimises the action functional — the ice-shelf rate of working under Glen's flow law — to find the velocity that satisfies the membrane stress balance.

### Adding noise

Real satellite velocity products carry measurement uncertainty. MEaSUREs and ITS_LIVE products typically report formal errors of order 1–10 m yr$^{-1}$ per component, depending on orbit geometry and coherence. We mimic this by adding independent Gaussian noise with standard deviation $\sigma_\text{obs} = 5$ m yr$^{-1}$ to each component of the modelled velocity at every mesh node. The noisy field `u_obs` is our pretend satellite observation.

In [ ]:
# Initial velocity guess: plug flow increasing downstream
u_in_speed = 100.0   # m/yr
u0 = firedrake.Function(V).interpolate(
    firedrake.as_vector((u_in_speed + 50.0 * x / Lx, 0.0))
)

model = icepack.models.IceShelf()
solver = icepack.solvers.FlowSolver(model, dirichlet_ids=[1])  # verify against your icepack version

# Forward solve with the true fluidity
u_true = solver.diagnostic_solve(
    velocity=u0,
    thickness=h,
    fluidity=A_true,
)  # verify against your icepack version

print("True velocity — max speed:", u_true.dat.data.max(), "m/yr")

# Add Gaussian noise to mimic satellite observations
sigma_obs = 5.0   # m/yr per component
rng = np.random.default_rng(seed=42)

u_obs = firedrake.Function(V)
u_obs.dat.data[:] = (
    u_true.dat.data
    + rng.normal(0.0, sigma_obs, u_true.dat.data.shape)
)

# Plot true speed and noisy observation
import firedrake
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (u_field, title) in zip(axes, [(u_true, 'True speed'), (u_obs, 'Observed speed (noisy)')]):
    ax.set_aspect('equal')
    import firedrake
    spd = firedrake.Function(Q).interpolate(firedrake.sqrt(firedrake.inner(u_field, u_field)))
    colors = firedrake.tripcolor(spd, axes=ax)
    fig.colorbar(colors, ax=ax, label='speed (m/yr)')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Part 3 — the inverse problem

### What we are trying to do

We want to find a log-fluidity field $\theta(x,y)$ such that the velocity $u$ obtained by solving the momentum balance with $A = A_0 e^\theta$ matches the observed velocity $u_\text{obs}$ as closely as possible. Written as an optimisation, we want to minimise

$$J(\theta) = \underbrace{\frac{1}{2\sigma_\text{obs}^2} \int_\Omega |u(\theta) - u_\text{obs}|^2\,\mathrm{d}x}_{\text{misfit}} + \underbrace{\frac{\alpha}{2} \int_\Omega |\nabla\theta|^2\,\mathrm{d}x}_{\text{regularisation}}.$$

### Why misfit alone is ill-posed

If you kept only the misfit term, the problem would be **ill-posed** in the sense of Hadamard: there are infinitely many $\theta$ fields that produce velocities matching the data at the level of noise, including fields with arbitrarily fine-scale oscillations that happen to cancel in the momentum balance. High-wavenumber perturbations to $\theta$ are damped out in the velocity signal — the momentum equation is a smoothing operator — so the data carry almost no information about them. Minimising the misfit alone amplifies whatever noise sits at those unresolvable scales.

### What the regularisation term does

The second term, the Tikhonov smoothness penalty, controls this by penalising spatial gradients in $\theta$. A larger $\alpha$ forces the recovered field to be smoother, at the cost of blurring the features we care about (the soft band). A smaller $\alpha$ allows finer structure but lets noise corrupt the solution. The regularisation parameter $\alpha$ is the principal tuning knob of the inversion, and choosing it well is as much physics as it is statistics — you should pick the smallest $\alpha$ for which the solution is still geophysically plausible, guided by the known or estimated noise level.

### The icepack estimator

icepack implements this minimisation through a `MaximumProbabilityEstimator` that uses the adjoint of the forward solver to compute gradients of $J$ with respect to $\theta$ automatically. You initialise it with the model, the solver, the observations, and the prior (which encodes $\alpha$), and it runs a gradient-based optimisation to convergence. Under the hood it is doing exactly what a research group would do with a hand-rolled adjoint model, but the adjoint is computed automatically by Firedrake's algorithmic differentiation.

The API shown below follows the pattern in the icepack inverse tutorial. The specific argument names and class structure have changed between icepack versions, so every call that touches `icepack.statistics` is flagged.

In [ ]:
# Starting guess for the log-fluidity: uniform zero (A = A0 everywhere)
theta_init = firedrake.Function(Q)
theta_init.assign(0.0)

# Regularisation parameter
alpha = 1e-3   # tune this in Task 1

# Observation and prior statistics objects
# These class names and keyword arguments should be verified against your
# installed icepack version — the tutorial at
# https://icepack.github.io/notebooks/inverse-problems/ is the authoritative reference.

obs = icepack.statistics.ObservationalDataset(  # verify against your icepack version
    velocity=u_obs,
    sigma=firedrake.Constant(sigma_obs),
)

prior = icepack.statistics.GaussianRegularisation(  # verify against your icepack version
    parameter=theta_init,
    regularisation_parameter=firedrake.Constant(alpha),
)

print("Observation and prior objects created.")

In [ ]:
# Build the estimator and run it
# The diagnostic_solve call inside the estimator must receive the
# same fixed fields (thickness, initial velocity guess) as the forward solve above.

estimator = icepack.statistics.MaximumProbabilityEstimator(  # verify against your icepack version
    model=model,
    dataset=obs,
    prior=prior,
    solver_kwargs=dict(
        dirichlet_ids=[1],
    ),
)

theta_estimated = estimator.solve(  # verify against your icepack version
    parameter=theta_init.copy(deepcopy=True),
    thickness=h,
    velocity=u0.copy(deepcopy=True),
    fluidity_name="fluidity",          # keyword depends on icepack version
    A0=A0,
)

print("Inversion complete.")

## Part 4 — examining the recovery

Three panels tell the full story: the true log-fluidity $\theta_\text{true}$, the estimated $\hat\theta$, and the pointwise error $\hat\theta - \theta_\text{true}$. A good inversion will recover the Gaussian band — its location, rough amplitude, and shape — while the error panel should show residuals that look like smooth noise, not a systematic miss of the band's peak.

We also compute the **misfit reduction**: how much has the RMS velocity error dropped from using the homogeneous starting guess ($\theta = 0$) to the estimated $\hat\theta$? A large reduction means the inversion has genuinely used the velocity information to locate the soft band.

In [ ]:
# Compute the estimated fluidity and re-solve for velocity
A_estimated = firedrake.Function(Q).interpolate(A0 * firedrake.exp(theta_estimated))

u_estimated = solver.diagnostic_solve(
    velocity=u0.copy(deepcopy=True),
    thickness=h,
    fluidity=A_estimated,
)  # verify against your icepack version

# Pointwise error in log-fluidity
theta_error = firedrake.Function(Q).interpolate(theta_estimated - theta_true)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
panels = [
    (theta_true,      r'$\theta_{\rm true}$',    None),
    (theta_estimated, r'$\hat{\theta}$',           None),
    (theta_error,     r'$\hat{\theta}-\theta_{\rm true}$', 'RdBu_r'),
]
for ax, (field, label, cmap_name) in zip(axes, panels):
    ax.set_aspect('equal')
    kwargs = {} if cmap_name is None else {"cmap": cmap_name}
    colors = firedrake.tripcolor(field, axes=ax, **kwargs)
    fig.colorbar(colors, ax=ax, label=label)
    ax.set_xlabel('x (m)')
plt.tight_layout()
plt.show()

# Misfit reduction
def rms_velocity_error(u_model, u_ref, Q_space):
    diff = firedrake.Function(Q_space).interpolate(
        firedrake.sqrt(firedrake.inner(u_model - u_ref, u_model - u_ref))
    )
    vals = diff.dat.data_ro
    return float(np.sqrt(np.mean(vals**2)))

# Baseline: forward solve with theta = 0 (A = A0 everywhere)
A_baseline = firedrake.Function(Q).interpolate(A0)
u_baseline = solver.diagnostic_solve(
    velocity=u0.copy(deepcopy=True),
    thickness=h,
    fluidity=A_baseline,
)  # verify against your icepack version

rms_baseline  = rms_velocity_error(u_baseline,  u_obs, Q)
rms_estimated = rms_velocity_error(u_estimated, u_obs, Q)

print(f"RMS velocity error — baseline (θ=0): {rms_baseline:.2f} m/yr")
print(f"RMS velocity error — estimated θ̂:   {rms_estimated:.2f} m/yr")
print(f"Misfit reduction: {100*(1 - rms_estimated/rms_baseline):.1f} %")

## Student tasks

### Task 1 — vary the regularisation parameter

Return to the inversion cell, change `alpha` to `1e-2` (ten times larger), re-run, and note what happens to the recovered soft band. Then try `alpha = 1e-4` (ten times smaller). For each value, record the misfit reduction and describe qualitatively whether the error field in the third panel looks systematic or noise-like. Draw a sketch of misfit-vs-alpha and mark roughly where you think the optimum lies.

If the large-alpha solution looks oversmoothed and the small-alpha solution looks noisy, you are seeing the **bias–variance trade-off** of regularisation in action. The goal is a solution that is the simplest one consistent with the data — not so smooth that it misses the real feature, not so rough that it is chasing noise.

In [ ]:
# Scaffold for Task 1: loop over three alpha values and collect results
alpha_values = [1e-4, 1e-3, 1e-2]
results_task1 = {}

for alpha_val in alpha_values:
    # YOUR CODE HERE:
    # 1. Re-create the prior with alpha = alpha_val
    # 2. Re-run the estimator (same obs, new prior)
    # 3. Re-solve for velocity with the new theta
    # 4. Compute and store the RMS misfit and the max absolute theta error
    #
    # results_task1[alpha_val] = dict(rms_misfit=..., max_error=...)
    pass

# YOUR CODE HERE: print a summary table and plot the three recovered
# theta fields side by side as above

### Task 2 — halve the noise

Re-create `u_obs` with `sigma_obs = 2.5` m yr$^{-1}$ (half the original), keep the default `alpha = 1e-3`, and re-run the inversion. Does the misfit reduction improve? Does the peak of the Gaussian band get recovered more accurately? Report the change in peak $\hat\theta$ versus the true peak and explain, in one sentence, the mechanism: why does less noise help the inversion even when the regularisation parameter is unchanged?

In [ ]:
# YOUR CODE HERE:
# 1. Regenerate u_obs with sigma_obs = 2.5 m/yr
# 2. Rebuild the ObservationalDataset with the new sigma
# 3. Re-run the estimator with alpha = 1e-3
# 4. Compare the peak recovered theta to theta_true.dat.data.max()
# 5. Compare RMS misfit to the Part 4 baseline
pass

### Task 3 — move the soft band

Change the Gaussian centre to $x_0 = 0.7\,L_x$ (the downstream third of the shelf) and repeat the inversion at the default settings. The ice is faster and thinner there. Does the inversion recover the band equally well as when it sat at $x_0 = 0.4\,L_x$? Think about why the sensitivity of velocity to a local fluidity perturbation might vary with position on the shelf — faster ice stretches more, so a given change in $A$ produces a larger velocity anomaly — and discuss whether that makes the downstream band easier or harder to find.

In [ ]:
# YOUR CODE HERE:
# 1. Change x0 to 0.7 * Lx and recompute theta_true and A_true
# 2. Re-run the forward solve to get the new u_true
# 3. Add noise at sigma_obs = 5.0 m/yr to get the new u_obs
# 4. Invert with alpha = 1e-3
# 5. Plot truth vs estimate vs error as in Part 4
# 6. Print the misfit reduction and peak theta recovery
pass

## Synthesis questions

Answer the following questions in a few sentences each; the goal is to connect the experiment you just ran to the broader problem of understanding ice sheets from remote sensing.

**Q1.** This experiment inverted surface velocity for fluidity in a floating ice shelf. The analogous problem for a grounded ice stream is inverting for **basal friction**, the parameter $C$ in a sliding law $\tau_b = C\,|u_b|^{1/m - 1} u_b$. Sketch, in words, how the workflow would change. The momentum balance is different (there is now basal drag alongside membrane stresses), but the structure of the objective function — misfit plus Tikhonov regularisation — stays the same. What is the equivalent of the soft band in the basal friction context, and why is inferring it even harder than the shelf problem here?

**Q2.** Velocity observations are described in the glaciology literature as "the currency of inversions." After running this lab, why do you think that is? What property of the momentum balance makes velocity sensitive to spatially averaged material properties, and what does that sensitivity imply about the spatial scales of features we can and cannot recover?

**Q3.** The regularisation parameter $\alpha$ was chosen by hand here. In real inversions a common criterion is the **L-curve** method: plot the misfit norm against the regularisation norm for a range of $\alpha$ values on a log–log scale and look for the corner where both start increasing rapidly. Why does such a corner typically exist, and what does it represent physically? Would you expect the L-curve corner to shift if you doubled the satellite velocity noise?

**Q4.** The experiment used a rectangular shelf with periodic, parallel flow. Real ice shelves — Pine Island, Thwaites, Filchner-Ronne — have irregular calving fronts, island pinning points, and pronounced lateral shear zones. Name one structural feature of real shelf geometry that would complicate the inversion and describe one way practitioners handle it.

## Further reading and technical notes

The ice-shelf stress balance underlying the forward model is derived in {cite}`macayeal1989`. The icepack software framework, including the automatic adjoint approach used by the estimator, is described in {cite}`shapero2021`. For Tikhonov regularisation in the broader geophysical inverse problem context, see [TODO-CITE: Aster-Borchers-Thurber parameter estimation], which is the standard graduate text. The first demonstration of the ice-shelf control method — inferring basal melting from surface velocity — is [TODO-CITE: MacAyeal-Rommelaere 1995 control method]; the modern adjoint formulation for marine ice sheets is given in [TODO-CITE: Goldberg-Heimbach 2013 adjoint icesheet]. For the satellite velocity products you would use in a real inversion, see the ITS_LIVE project [TODO-CITE: Gardner ITS-LIVE] and MEaSUREs [TODO-CITE: Mouginot MEaSUREs velocities].

**On icepack API stability.** The `icepack.statistics` module has evolved across icepack versions. Lines that depend on its API are marked `# verify against your icepack version` throughout this notebook. The most reliable reference is the inverse-problems tutorial distributed with your icepack installation: run `python -c "import icepack; print(icepack.__version__)"` to identify your version and consult the matching tutorial at `https://icepack.github.io`.